# Analyse Diya data

Uses the cell, medium and optical-power maps from `plate_reader_analysis_diya_kirill_m9.py` and the three `26-10-21 Kirill M9_diya_extracted_*.csv` files in `Yuhang/Data`.

This is one plate all wells receive red light (2.8 in the source map), and different dosages of green light. Samples are grouped by label and green-light intensity. Blank wells are pooled across all green-light intensities for each control label, and the same pooled blank is used for every light condition.

Run all cells in order. Edit **Settings** to select channels, samples, lights, plot types and exports. By default, both refreshed (`_D`) and stationary (`_O`) samples are shown. `K` denotes single plasmid and `N` double plasmid control. Suffixes 1–3 denote biological replicates, each with three technical wells per light. Biological replicates remain separate curves.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.embed import file_html
from IPython.display import HTML, display
from bokeh.resources import CDN, INLINE
from bokeh.models import FactorRange, ColumnDataSource
import html as html_lib

# Works when launched from the repository root or any directory beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Yuhang/plate_reader_analysis_diya_kirill_lb.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Launch this notebook from within the CcaSolitaiRe repository.')
DATA_DIR = ROOT / 'Yuhang/Data'

## Settings

Blank wells are pooled across all light intensities for each control label and averaged per time point and measurement channel. The same pooled blank is used for every light condition. `BLANK_LABEL_BY_FAMILY` defines the assignments; `BLANK_WELLS` optionally selects a subset of each control's wells using label keys (e.g. `{'B': ['H10', 'H11', 'H12']}`). Wells labelled `media` are not used.

`EXCLUDED_MEASUREMENTS` masks individual sample readings before averaging. Keys are `(light, well)` and values are zero-based reading indices. Summary plots omit the initial reading by default, as in `014_testing.ipynb`; set `SUMMARY_START_INDEX = 0` to include it. Replicate and blank plots show all readings.

In [2]:
LIGHTS = [0.0, 0.028, 0.28, 0.56, 1.4]
PROBE_CHANNEL = 'GFP 395nm'              # Or 'GFP 488nm'
SELECTED_SAMPLES = [f'{family}_{rep}' for family in ['K_D', 'N_D']
                    for rep in [1, 2, 3]]
PLOTS = ['replicates', 'blanks', 'summary']
SUMMARY_METRICS = ['normalised_fluorescence', 'od600']
SUMMARY_START_INDEX = 1                  # Set 0 to include the initial reading
Y_RANGES = {'normalised_fluorescence': None, 'od600': None}  # e.g. (0, 1000)
BLANK_LABEL_BY_FAMILY = {'K_D': 'B', 'N_D': 'B', 'W_D': 'B'} # here, a common blank for all families
BLANK_WELLS = {}                         # All wells per label; e.g. {'B': ['H10', 'H11', 'H12']}
EXCLUDED_MEASUREMENTS = {}
SAVE_HTML = False
OUTPUT_DIR = ROOT / 'Yuhang/26-10-21 Kirill M9/figures_diya_analysis'


## Plate maps

Arrays and labels copied from the Diya script. Light labels derived from the per-well optical-power arrays.

In [3]:
NCOLS, NROWS = 12, 8
CELL_NAMES = {0: 'null',
              1: 'W_D_1', 2: 'N_D_1', 3: 'K_D_1',
              4: 'W_D_2', 5: 'N_D_2', 6: 'K_D_2',
              7: 'W_D_3', 8: 'N_D_3', 9: 'K_D_3',
              10: 'B'}

cell_map = np.array([
    #1   2   3   4   5   6   7   8   9   10  11  12
    [1,  2,  3,  4,  5,  6,  7,  8,  9,   1,  2,  3],  # Row A
    [1,  2,  3,  4,  5,  6,  7,  8,  9,   1,  2,  3],  # Row B
    [1,  2,  3,  4,  5,  6,  7,  8,  9,   1,  2,  3],  # Row C
    [1,  2,  3,  4,  5,  6,  7,  8,  9,   1,  2,  3],  # Row D
    [1,  2,  3,  4,  5,  6,  7,  8,  9,   1,  2,  3],  # Row E
    [8,  9,  4,  5,  6,  7,  8,  9,  4,   5,  6,  7],  # Row F
    [8,  9,  4,  5,  6,  7,  8,  9,  4,   5,  6,  7],  # Row G
    [10,10,  4,  5,  6,  7,  8,  9, 10,  10, 10, 10],  # Row H
])

OPTICAL_POWER = np.array([

	# Channel 0 (Color 0 or 4). Blue on v0.4c
	[[0 / (2**(7-row)) for col in range(NCOLS)] for row in range(NROWS)],

    # Channel 1 – Green light
    np.array([
        # Columns 1–12 (A–L), Rows A–H
        [0.0,		0.56,   	1.4,    	0.028,	0.56,	0.028,  0.0,    1.4,	0.56,   0.28,   0.0,	0.028],  # Row A
        [0.028,  	1.4,  		0.56,  		0.0,	1.4,  	0.0,  	0.56,	0.028,  0.28,  	0.028,	1.4,  	0.56],  # Row B
        [0.28,  	0.28,  		0.0,  		0.28,	0.028,  1.4,  	0.28,  	0.56,  	1.4,  	0.0,  	0.56,  	0.28],  # Row C
        [1.4,		0.028,  	0.028,    	1.4,	0.0,	0.56,  	0.028,  0.0,	0.028,  0.56,   0.28,	1.4],  # Row D
        [0.56,  	0.0,  		0.28,  		0.56,	0.28,  	0.28,  	1.4, 	0.28,  	0.0,  	1.4,    0.028,  0.0],  # Row E
        [0.56,		0.0,    	0.0,    	0.28,	0.028,	0.56,   1.4,  	0.56,	1.4,    0.0,  	0.56,	0.028],  # Row F
        [0.028,		0.28,    	0.028,  	1.4,	0.28,	1.4,    0.0, 	1.4,	0.56,   0.56,  	1.4,	0.0],  # Row G
        [0.56,		0.56,    	0.28,    	0.028,	0.0,    0.28,   0.28,   0.028,  0.56,   0.0,    0.0,    0.0],  # Row H
    ]),
	# Channel 2 (Color 2 or 6). Yellow-Green or White on v0.4c
	[[0 / (2**row) for col in range(NCOLS)] for row in range(NROWS)],

    # Channel 3 – Red light
    np.array([
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row A
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row B
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row C
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row D
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row E
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row F
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row G
        [2.8,	2.8,	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,  	2.8,    2.8],  # Row H
    ])
])

In [4]:
# EXCLUDE ALL N_D_3 MEASUREMENTS (bad clone of two-plasmid CcaSR)
# Removing the sample excludes every well and reading from analysis and plots.
SELECTED_SAMPLES = [sample for sample in SELECTED_SAMPLES if sample != 'N_D_3']


Gather all the information into a single plate_map dataframe

In [5]:
# get well labels and sample identities
plate_map = pd.DataFrame([
    {'well': f'{row}{c+1}', 'sample': CELL_NAMES[int(cell_map[r,c])],
     'row': row, 'column': c+1}
    for r, row in enumerate('ABCDEFGH') for c in range(12)
]).set_index('well')

# verify that the plate map has a unique index
assert len(plate_map) == 96 and plate_map.index.is_unique

# get green and red light intensities
plate_map['green_intensity'] = OPTICAL_POWER[1].reshape(-1)
plate_map['red_intensity'] = OPTICAL_POWER[3].reshape(-1)
plate_map['light'] = plate_map['green_intensity'] # overall lighting differs only by green light condition

# verify that all the selected samples are included in the plate map
unknown = set(SELECTED_SAMPLES) - set(plate_map['sample'])
if unknown:
    raise ValueError(f'Samples absent from this plate map: {unknown}')
def sample_family(sample):
    return sample.rsplit('_', 1)[0] if sample.rsplit('_', 1)[-1].isdigit() else sample

# get blanks
## match each sample to the corresponding blanks
sample_blank = {}
for sample in SELECTED_SAMPLES:
    family = sample_family(sample)
    if family not in BLANK_LABEL_BY_FAMILY:
        raise ValueError(f'Assign a blank control in BLANK_LABEL_BY_FAMILY for {family}')
    sample_blank[sample] = BLANK_LABEL_BY_FAMILY[family]
blank_labels = list(dict.fromkeys(sample_blank.values()))
## pool blanks by control label across all light intensities
unknown_blank_labels = set(BLANK_WELLS) - set(blank_labels)
if unknown_blank_labels:
    raise ValueError(f'BLANK_WELLS keys must be control labels from {blank_labels}; got {unknown_blank_labels}')
pooled_blank_wells = {}
for label in blank_labels:
    available_wells = plate_map.index[plate_map['sample'].eq(label)].tolist()
    wells = list(BLANK_WELLS.get(label, available_wells))
    if not wells or len(set(wells)) != len(wells):
        raise ValueError(f'Blank wells for {label} must be nonempty and unique.')
    if not set(wells) <= set(available_wells):
        raise ValueError(f'Blank wells for {label} must match the control label in the map.')
    pooled_blank_wells[label] = wells

# Use the same pooled wells for each light condition.
blank_wells = {light: {label: wells.copy() for label, wells in pooled_blank_wells.items()}
               for light in LIGHTS}

# verify that each sample has been tested in each light condition
def sample_indices(sample, light):
    return np.flatnonzero(plate_map['sample'].eq(sample) & plate_map['light'].eq(light))
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        if not len(sample_indices(sample, light)):
            raise ValueError(f'No wells for {sample}, {light}')

# show the plate map for verification
display(plate_map.pivot(index='row', columns='column', values='sample'))
display(plate_map.pivot(index='row', columns='column', values='light'))

# show how samples are matched to their blanks
for label, wells in pooled_blank_wells.items():
    print(f'{label} pooled blank wells (all light conditions): {", ".join(wells)}')

column,1,2,3,4,5,6,7,8,9,10,11,12
row,,,,,,,,,,,,
A,W_D_1,N_D_1,K_D_1,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_1,N_D_1,K_D_1
B,W_D_1,N_D_1,K_D_1,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_1,N_D_1,K_D_1
C,W_D_1,N_D_1,K_D_1,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_1,N_D_1,K_D_1
D,W_D_1,N_D_1,K_D_1,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_1,N_D_1,K_D_1
E,W_D_1,N_D_1,K_D_1,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_1,N_D_1,K_D_1
F,N_D_3,K_D_3,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_2,N_D_2,K_D_2,W_D_3
G,N_D_3,K_D_3,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,W_D_2,N_D_2,K_D_2,W_D_3
H,B,B,W_D_2,N_D_2,K_D_2,W_D_3,N_D_3,K_D_3,B,B,B,B


column,1,2,3,4,5,6,7,8,9,10,11,12
row,,,,,,,,,,,,
A,0.000,0.560,1.400,0.028,0.560,0.028,0.000,1.400,0.560,0.280,0.000,0.028
B,0.028,1.400,0.560,0.000,1.400,0.000,0.560,0.028,0.280,0.028,1.400,0.560
C,0.280,0.280,0.000,0.280,0.028,1.400,0.280,0.560,1.400,0.000,0.560,0.280
D,1.400,0.028,0.028,1.400,0.000,0.560,0.028,0.000,0.028,0.560,0.280,1.400
E,0.560,0.000,0.280,0.560,0.280,0.280,1.400,0.280,0.000,1.400,0.028,0.000
F,0.560,0.000,0.000,0.280,0.028,0.560,1.400,0.560,1.400,0.000,0.560,0.028
G,0.028,0.280,0.028,1.400,0.280,1.400,0.000,1.400,0.560,0.560,1.400,0.000
H,0.560,0.560,0.280,0.028,0.000,0.280,0.280,0.028,0.560,0.000,0.000,0.000


B pooled blank wells (all light conditions): H1, H2, H9, H10, H11, H12


# Note: using OD600_corrected file. For H9-H12 balnks, t=24h entries are duplicated penultimate entries as in reality they had been contaminated

Refer to files with `_orig` suffixes to see original data

In [6]:
# initialise raw/processed measurements, measurement times, and blank measuremenys
raw, times, blanks, processed = {}, {}, {}, {}
acquisition_rows, qc_rows = [], []

# read measurements by light condition and by observation channel
for light in LIGHTS:
    raw[light] = {}
    for channel in ['OD600', PROBE_CHANNEL]:
        # read the file
        path = DATA_DIR / f'26-10-21 Kirill M9_diya_extracted_{channel}.csv'
        frame = pd.read_csv(path, index_col=0)

        # verify that there are records for all 96 well plates
        if not frame.index.is_unique or set(frame.index) != set(plate_map.index):
            raise ValueError(f'Expected exactly 96 unique wells in {path}')

        # get measurement times
        frame = frame.loc[plate_map.index].astype(float)
        stamps = pd.to_datetime(frame.columns, dayfirst=True, format='mixed')
        if not stamps.is_monotonic_increasing or not stamps.is_unique:
            raise ValueError(f'Timestamps must be unique and increasing: {path}')

        # veri that measurements are finite
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f'Non-finite measurements in {path}')

        # get raw measuremenet
        raw[light][channel] = frame

        # get the labels for acquired well
        for i, stamp in enumerate(stamps):
            acquisition_rows.append({'light': light, 'channel': channel, 'reading': i, 'timestamp': stamp})

        # it this is an OD measurement, set its time of acquisition for all measured variables
        if channel == 'OD600':
            times[light] = ((stamps - stamps[0]).total_seconds() / 3600).to_numpy()

    # get raw data for all samples
    od_raw = raw[light]['OD600'].to_numpy()
    flu_raw = raw[light][PROBE_CHANNEL].to_numpy()
    if od_raw.shape != flu_raw.shape:
        raise ValueError(f'Different numbers of OD and fluorescence data points for {light}')
    # get mean blank ODs
    blanks[light] = {
        label: {channel: frame.loc[wells].mean(axis=0).to_numpy()
                for channel, frame in raw[light].items()}
        for label, wells in blank_wells[light].items()
    }

    # normalise measurements for selected samples; blanks/media/unselected samples get nans
    ## initialize with nans
    processed[light] = {metric: np.full_like(od_raw, np.nan)
                        for metric in ['od600', 'normalised_fluorescence']}
    for sample in SELECTED_SAMPLES:
        idx = sample_indices(sample, light)
        label = sample_blank[sample]
        blank = blanks[light][label]
        # normalise the OD and floor it
        od_delta = od_raw[idx] - blank['OD600']
        od = np.maximum(od_delta, 0.005)

        # normalise the fluorescence and floor it
        normalised = np.maximum((flu_raw[idx] - blank[PROBE_CHANNEL]) / od, 0)

        # gather the processed data
        processed[light]['od600'][idx] = od
        processed[light]['normalised_fluorescence'][idx] = normalised
        qc_rows.append({'light': light, 'sample': sample, 'blank_control': label,
                        'blank_wells': ','.join(blank_wells[light][label]),
                        'technical_wells': len(idx),
                        'OD_values_floored': int((od_delta < 0.005).sum())})

# show the labels and time stamps for all acquired data
display(pd.DataFrame(acquisition_rows))
display(pd.DataFrame(qc_rows))

,light,channel,reading,timestamp
0,0.0,OD600,0,2026-10-20 11:12:00
1,0.0,OD600,1,2026-10-20 14:29:00
2,0.0,OD600,2,2026-10-20 17:41:00
3,0.0,OD600,3,2026-10-20 20:37:00
4,0.0,OD600,4,2026-10-20 23:23:00
...,...,...,...,...
65,1.4,GFP 395nm,2,2026-10-20 17:42:00
66,1.4,GFP 395nm,3,2026-10-20 20:38:00
67,1.4,GFP 395nm,4,2026-10-20 23:24:00
68,1.4,GFP 395nm,5,2026-10-21 02:17:00


,light,sample,blank_control,blank_wells,technical_wells,OD_values_floored
0,0.000,K_D_1,B,"H1,H2,H9,H10,H11,H12",2,2
1,0.000,K_D_2,B,"H1,H2,H9,H10,H11,H12",2,2
2,0.000,K_D_3,B,"H1,H2,H9,H10,H11,H12",2,2
3,0.000,N_D_1,B,"H1,H2,H9,H10,H11,H12",2,2
4,0.000,N_D_2,B,"H1,H2,H9,H10,H11,H12",2,2
5,0.028,K_D_1,B,"H1,H2,H9,H10,H11,H12",2,1
6,0.028,K_D_2,B,"H1,H2,H9,H10,H11,H12",2,2
7,0.028,K_D_3,B,"H1,H2,H9,H10,H11,H12",2,2
8,0.028,N_D_1,B,"H1,H2,H9,H10,H11,H12",2,2
9,0.028,N_D_2,B,"H1,H2,H9,H10,H11,H12",2,2


In [7]:
# Masks are applied to samples, not to the blank averages.
masks = {light: np.ones_like(processed[light]['od600'], dtype=bool) for light in LIGHTS}
for (light, well), indices in EXCLUDED_MEASUREMENTS.items():
    if light not in masks or well not in plate_map.index:
        raise ValueError(f'Unknown mask target: {light}, {well}')
    if plate_map.loc[well, 'light'] != light or plate_map.loc[well, 'sample'] not in SELECTED_SAMPLES:
        raise ValueError(f'Mask target must be a selected sample under {light}: {well}')
    if any(i < 0 or i >= len(times[light]) for i in indices):
        raise ValueError(f'Invalid reading index for {light}, {well}')
    masks[light][plate_map.index.get_loc(well), indices] = False

# Gather the data into a summary dataframe
summary_rows = []
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        idx = sample_indices(sample, light)
        for metric, values in processed[light].items():
            for t, hours in enumerate(times[light]):
                included = values[idx, t][masks[light][idx, t]]
                summary_rows.append(dict(light=light, sample=sample, metric=metric,
                    blank_control=sample_blank[sample], reading=t, hours=hours, n=len(included),
                    mean=included.mean() if len(included) else np.nan,
                    sd=included.std(ddof=0) if len(included) else np.nan))
summary = pd.DataFrame(summary_rows)
# show a representatie entry in the summary dataframe
display(summary.head(12))

,light,sample,metric,blank_control,reading,hours,n,mean,sd
0,0.0,K_D_1,od600,B,0,0.000000,2,0.005000,0.000000
1,0.0,K_D_1,od600,B,1,3.283333,2,0.028017,0.001500
2,0.0,K_D_1,od600,B,2,6.483333,2,0.296500,0.025500
3,0.0,K_D_1,od600,B,3,9.416667,2,0.331400,0.029050
4,0.0,K_D_1,od600,B,4,12.183333,2,0.372883,0.056800
5,0.0,K_D_1,od600,B,5,15.050000,2,0.374783,0.024850
6,0.0,K_D_1,od600,B,6,24.400000,2,0.440267,0.028100
7,0.0,K_D_1,normalised_fluorescence,B,0,0.000000,2,1200.000000,1200.000000
8,0.0,K_D_1,normalised_fluorescence,B,1,3.283333,2,118.577097,118.577097
9,0.0,K_D_1,normalised_fluorescence,B,2,6.483333,2,202.188052,24.134217


## Plots

Replicate panels show each technical well plus its masked mean. Blank panels show raw selected blank wells and their mean, before OD flooring. Summary panels group `K_D`, `N_D`, etc., retain a separate curve for each biological replicate, and show ±1 SD across its technical wells. Click legend entries to hide curves. Bokeh's toolbar supports zoom, pan, reset and image saving; `SAVE_HTML` also writes interactive standalone HTML files.

Plot measurements:

In [8]:
# specify plotting styles
COLOURS = {0.0: 'crimson', 0.028: 'darkgoldenrod', 0.28: 'darkkhaki', 0.56: 'mediumseagreen', 1.4: 'darkgreen'}
DASHES = ['solid', 'dashed', 'dotted', 'dashdot']
LABELS = {'od600': 'Blanked OD600',
          'normalised_fluorescence': f'{PROBE_CHANNEL} / OD600 (blanked, AU)'}

# figure creation function
def new_figure(title, ylabel, metric=None):
    options = {} if Y_RANGES.get(metric) is None else {'y_range': Y_RANGES[metric]}
    return figure(title=title, x_axis_label='Time since first OD reading, h',
                  y_axis_label=ylabel, width=400, height=300,
                  tools='pan,wheel_zoom,box_zoom,reset,save', **options)

#  show all plots after styling the legends
def emit(layout, name, legend_margin=1):
    from bokeh.models import Legend
    for legend in layout.select({'type': Legend}):
        legend.background_fill_color = 'white'
        legend.background_fill_alpha = 1.0
        legend.border_line_color = 'black'
        legend.border_line_alpha = 1.0
        legend.margin = legend_margin
        legend.spacing = 1
        legend.padding = 1
    html_doc = file_html(layout, CDN, 'Plots')
    display(HTML(f'<iframe srcdoc="{html_lib.escape(html_doc)}" width="100%" height="700" style="border:none;"></iframe>'))
    if SAVE_HTML:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        channel_slug = PROBE_CHANNEL.replace(' ', '_')
        target = OUTPUT_DIR / f'{channel_slug}_{name}.html'
        target.write_text(file_html(layout, INLINE, name), encoding='utf-8')
        print(f'Saved {target}')

# plot data for different replicates
if 'replicates' in PLOTS:
    for metric in SUMMARY_METRICS:
        panels = []
        for sample in SELECTED_SAMPLES:
            for light in LIGHTS:
                p = new_figure(f'{sample}, {light}', LABELS[metric], metric)
                idx = sample_indices(sample, light)
                for j, row in enumerate(idx):
                    values = np.where(masks[light][row], processed[light][metric][row], np.nan)
                    p.line(times[light], values, color=COLOURS[light], alpha=0.35,
                           line_dash=DASHES[j % 3], legend_label=plate_map.index[row])
                s = summary.query('light == @light and sample == @sample and metric == @metric')
                p.line(s.hours, s['mean'], color=COLOURS[light], line_width=3, legend_label='Mean')
                p.legend.location = 'right'
                p.legend.click_policy = 'hide'
                panels.append(p)
        if panels:
            emit(gridplot(panels, ncols=len(LIGHTS)), f'replicates_{metric}')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [9]:
if 'blanks' in PLOTS:
    panels = []
    for light in LIGHTS:
        for label, wells in blank_wells[light].items():
            for channel in ['OD600', PROBE_CHANNEL]:
                p = new_figure(f'{light}: {label} ({", ".join(wells)})', f'Raw {channel}')
                for well in wells:
                    p.line(times[light], raw[light][channel].loc[well].to_numpy(),
                           color=COLOURS[light], alpha=0.35, legend_label=well)
                p.line(times[light], blanks[light][label][channel], color='black',
                       line_width=3, legend_label='Control mean')
                p.legend.padding = 0
                p.legend.margin = 2
                p.legend.spacing = 0
                p.legend.click_policy = 'hide'
                panels.append(p)
    emit(gridplot(panels, ncols=2), 'blanks')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Plot means

In [10]:
if 'summary' in PLOTS:
    families = list(dict.fromkeys(s.rsplit('_', 1)[0] if s.rsplit('_', 1)[-1].isdigit() else s
                                 for s in SELECTED_SAMPLES))
    for metric in SUMMARY_METRICS:
        panels = []
        for family in families:
            p = new_figure(f'{family}: technical means ± SD', LABELS[metric], metric)
            samples = [s for s in SELECTED_SAMPLES if s == family or s.rsplit('_', 1)[0] == family]
            for j, sample in enumerate(samples):
                for light in LIGHTS:
                    s = summary.query('light == @light and sample == @sample and metric == @metric').copy()
                    s = s[s.reading >= SUMMARY_START_INDEX]
                    label = f'{sample}, {light}'
                    p.segment(s.hours, s['mean'] - s.sd, s.hours, s['mean'] + s.sd,
                              color=COLOURS[light], alpha=0.5, legend_label=label)
                    p.line(s.hours, s['mean'], color=COLOURS[light], line_width=2,
                           line_dash=DASHES[j % 3], legend_label=label)
                    p.scatter(s.hours, s['mean'], color=COLOURS[light],
                              marker=['plus', 'square', 'circle'][j % 3], size=7, legend_label=label)
            legend = p.legend[0]
            p.center.remove(legend)
            p.add_layout(legend, 'right')
            legend.location = 'top_left'
            legend.click_policy = 'hide'
            legend.label_text_font_size = '8pt'
            panels.append(p)

        emit(gridplot(panels, ncols=2), metric, legend_margin=12)

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [11]:
# Fit the blank-corrected OD600 for each biological replicate and light.
from scipy.optimize import curve_fit
# logistic growth function
def logistic_growth(t, r, Nmax, N0):
    return Nmax / (1 + ((Nmax - N0) / N0) * np.exp(-r * t))

# fit logistic growth curve for each biological replicate and light
logistic_params = {}
for sample in SELECTED_SAMPLES:
    logistic_params[sample] = {}
    for light in LIGHTS:
        idx = sample_indices(sample, light)
        # get ods and times for each technical replicate, collate all into one array
        od_points = []
        time_points = []
        for j, row in enumerate(idx):
            od_points.append(np.where(masks[light][row], processed[light]['od600'][row], np.nan))
            time_points.append(np.where(masks[light][row], times[light], np.nan))
        # flatten all points into one array
        time_points=(np.asarray(time_points)).flatten()
        od_points=(np.asarray(od_points)).flatten()
        # fit the logistic curve
        valid_points = np.isfinite(time_points) & np.isfinite(od_points)
        r, Nmax, N0 = curve_fit(logistic_growth, time_points[valid_points], od_points[valid_points])[0]
        logistic_params[sample][light] = {'r': r, 'Nmax': Nmax, 'N0': N0}

# PLOT LOGISTIC GROWTH CURVES
panels = []
for sample in SELECTED_SAMPLES:
    for light in LIGHTS:
        # get logistic estimates
        s = summary.query('light == @light and sample == @sample and metric == @metric').copy()
        t_sample_light = np.linspace(0.0, s['hours'].max(),100)
        r, Nmax, N0 = logistic_params[sample][light]['r'], logistic_params[sample][light]['Nmax'], logistic_params[sample][light]['N0']
        od_sample_light = logistic_growth(t_sample_light, r, Nmax, N0)

        # plot
        p = new_figure(f'{sample}, {light}', LABELS[metric], metric)
        idx = sample_indices(sample, light)
        for j, row in enumerate(idx):
            values = np.where(masks[light][row], processed[light][metric][row], np.nan)
            p.line(times[light], values, color=COLOURS[light], alpha=0.35,
                   line_dash=DASHES[j % 3], legend_label=plate_map.index[row])
        s = summary.query('light == @light and sample == @sample and metric == @metric')
        p.line(t_sample_light, od_sample_light, color=COLOURS[light], line_width=3, legend_label='Logistic curve')
        p.legend.location = 'right'
        p.legend.click_policy = 'hide'
        panels.append(p)
emit(gridplot(panels, ncols=2), 'Logistic fit')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [12]:
# FIND EXPRESSION FOLD CHANGES RELATIVE TO ZERO GREEN LIGHT
max_green = max(LIGHTS)
fold_rows = []

for light in LIGHTS:
    processed[light]['fold_change'] = np.full_like(
        processed[light]['normalised_fluorescence'], np.nan,
    )
    for sample in SELECTED_SAMPLES:
        # Pair technical wells in plate-map order within each sample and light.
        red_idx = sample_indices(sample, 0.0)
        current_idx = sample_indices(sample, light)
        red_values = processed[0.0]['normalised_fluorescence'][red_idx]
        current_values = processed[light]['normalised_fluorescence'][current_idx]
        if red_values.shape != current_values.shape:
            raise ValueError(f'Cannot pair technical wells/readings for {sample}, {light} with zero green light.')
        valid = (masks[0.0][red_idx] & masks[light][current_idx]
                 & np.isfinite(red_values) & np.isfinite(current_values)
                 & (red_values > 0))
        folds = np.divide(
            current_values, red_values,
            out=np.full_like(current_values, np.nan), where=valid,
        )
        processed[light]['fold_change'][current_idx] = folds

        for t, hours in enumerate(times[light]):
            included = folds[:, t][np.isfinite(folds[:, t])]
            fold_rows.append(dict(
                light=light, sample=sample, metric='fold_change',
                blank_control=sample_blank[sample], reading=t, hours=hours,
                n=len(included),
                mean=included.mean() if len(included) else np.nan,
                sd=included.std(ddof=0) if len(included) else np.nan,
            ))

# Replace previous fold-change rows so this cell can be rerun safely.
summary = pd.concat([
    summary.loc[summary['metric'].notna() & summary['metric'].ne('fold_change')],
    pd.DataFrame(fold_rows),
], ignore_index=True)
# Remove numbered columns left by the previous raw-array concatenation.
summary = summary.loc[:, ['light', 'sample', 'metric', 'blank_control',
                         'reading', 'hours', 'n', 'mean', 'sd']]


In [16]:
# PLOT PROTEIN LEVELS 6H SINCE THE START OF CULTURING
from bokeh.models import FactorRange, ColumnDataSource

metric = 'normalised_fluorescence'
s = summary.query('metric == @metric').copy()
s = s[
    s['sample'].isin(SELECTED_SAMPLES)
    & s['light'].isin(LIGHTS)
    & (s['n'] > 0)
    & np.isfinite(s['mean'])
].copy()
if s.empty:
    raise ValueError('No fluorescence values in summary; run the correction and summary cells first.')

# PLOT EXPRESSION
# Use the recorded time closest to 6 h, then average the biological replicate means
BAR_TARGET_HOURS = 6
bar_hours = s.loc[(s['hours'] - BAR_TARGET_HOURS).abs().idxmin(), 'hours']
s = s[s['hours'] == bar_hours].copy()
s['family'] = s['sample'].map(sample_family)
bar_stats = s.groupby(['family', 'light'])['mean'].agg(
    mean='mean', sd=lambda values: values.std(ddof=0), n='count',
).reset_index()
families = list(dict.fromkeys(sample_family(sample) for sample in SELECTED_SAMPLES))
max_green = max(LIGHTS)
bar_lights = [light for light in [0.0, max_green] if light in LIGHTS]
factors = [(family, str(light)) for family in families for light in bar_lights
           if ((bar_stats['family'] == family) & (bar_stats['light'] == light)).any()]

p_exp = figure(title=f'Normalised fluorescence at {bar_hours:.0f} h: biological means ± SD',
           x_range=FactorRange(*factors), x_axis_label='Sample family / light condition',
           y_axis_label=LABELS[metric], width=400, height=300,
           y_axis_type='log',
           tools='pan,wheel_zoom,box_zoom,reset,save')
for light in bar_lights:
    s = bar_stats[bar_stats['light'] == light]
    source = ColumnDataSource(dict(
        x=[(family, str(light)) for family in s['family']],
        mean=s['mean'].to_numpy(),
        lower=(s['mean'] - s['sd']).to_numpy(),
        upper=(s['mean'] + s['sd']).to_numpy(),
    ))
    p_exp.vbar(x='x', top='mean', bottom=1, width=0.75, source=source,
           color=COLOURS[light], legend_label=str(light))
    p_exp.segment(x0='x', y0='lower', x1='x', y1='upper', source=source,
              color='black', line_width=2)
p_exp.y_range.start = 1e1
p_exp.xgrid.grid_line_color = None
p_exp.legend.location = 'top_left'
p_exp.legend.click_policy = 'hide'

# PLOT DYNAMIC RANGE (FOLD-CHANGE IN EXPRESSION)
# get max green light
max_green = max(LIGHTS)
metric = 'fold_change'
s = summary.query('metric == @metric').copy()
s = s[
    s['sample'].isin(SELECTED_SAMPLES)
    & s['light'].isin([max_green])
    & (s['n'] > 0)
    & np.isfinite(s['mean'])
].copy()
if s.empty:
    raise ValueError('No green-light fold-change values in summary; run the correction and summary cells first.')

s = s[s['hours'] == bar_hours].copy()
s['family'] = s['sample'].map(sample_family)
fold_stats = s.groupby(['family', 'light'])['mean'].agg(
    mean='mean', sd=lambda values: values.std(ddof=0), n='count',
).reset_index()
fold_lights = [max_green]
fold_families = [family for family in families if (fold_stats['family'] == family).any()]

p_fold = figure(title=f'Dynamic range (from data at {bar_hours:.0f} h): mean ± SD',
                x_range=FactorRange(*fold_families), x_axis_label='Sample family',
                y_axis_label='Dynamic range (green:red fold-change)', width=400, height=300,
                y_axis_type='log',
                tools='pan,wheel_zoom,box_zoom,reset,save',
                tooltips=[('Family', '@x'), ('Green / red', '@mean{0.00}'), ('SD', '@sd{0.00}')])
for light in fold_lights:
    s = fold_stats[fold_stats['light'] == light]
    source = ColumnDataSource(dict(
        x=s['family'].tolist(),
        mean=s['mean'].to_numpy(),
        lower=(s['mean'] - s['sd']).to_numpy(),
        upper=(s['mean'] + s['sd']).to_numpy(),
        sd=s['sd'].to_numpy(),
    ))
    p_fold.vbar(x='x', top='mean', bottom=1, width=0.75, source=source,
                color='gray')
    p_fold.segment(x0='x', y0='lower', x1='x', y1='upper', source=source,
                   color='black', line_width=2)
p_fold.y_range.start = 1
p_fold.y_range.end = max(100, (fold_stats['mean'] + fold_stats['sd']).max() * 1.15)
p_fold.xgrid.grid_line_color = None

# show plots
emit(gridplot([p_exp, p_fold], ncols=2), '6h_bars')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [14]:
# PLOT GROWTH RATES AS ESTIMATED BY LOGISTIC CURVE FITS
from bokeh.models import FactorRange, ColumnDataSource


# Each biological replicate contributes one fitted r per light condition.
growth_rates = pd.DataFrame([
    {'sample': sample, 'family': sample_family(sample), 'light': light,
     'r': logistic_params[sample][light]['r']}
    for sample in SELECTED_SAMPLES
    if sample_family(sample) in ['K_D', 'N_D']
    for light in LIGHTS
])
if growth_rates.empty:
    raise ValueError('Select K_D or N_D samples and run the logistic fitting cell first.')

# Match p_exp: population SD across biological replicates (ddof=0).
growth_stats = growth_rates.groupby(['family', 'light'])['r'].agg(
    mean='mean', sd=lambda values: values.std(ddof=0), n='count',
).reset_index()
growth_families = [family for family in ['K_D', 'N_D']
                   if family in growth_stats['family'].values]
growth_lights = list(LIGHTS)
growth_factors = [(family, str(light)) for family in growth_families for light in growth_lights
                  if ((growth_stats['family'] == family)
                      & (growth_stats['light'] == light)).any()]

p_growth = figure(title='Logistic growth rate: biological means ± SD',
                  x_range=FactorRange(*growth_factors),
                  x_axis_label='Sample family / light condition',
                  y_axis_label='Logistic growth rate r (h⁻¹)', width=700, height=300,
                  tools='pan,wheel_zoom,box_zoom,reset,save',
                  tooltips=[('Family / light', '@x'), ('Mean r', '@mean{0.000}'),
                            ('SD', '@sd{0.000}'), ('Biological replicates', '@n')])
for light in growth_lights:
    stats = growth_stats[growth_stats['light'] == light]
    source = ColumnDataSource(dict(
        x=[(family, str(light)) for family in stats['family']],
        mean=stats['mean'].to_numpy(),
        lower=(stats['mean'] - stats['sd']).to_numpy(),
        upper=(stats['mean'] + stats['sd']).to_numpy(),
        sd=stats['sd'].to_numpy(),
        n=stats['n'].to_numpy(),
    ))
    p_growth.vbar(x='x', top='mean', bottom=0, width=0.75, source=source,
                  color=COLOURS[light], legend_label=str(light))
    p_growth.segment(x0='x', y0='lower', x1='x', y1='upper', source=source,
                     color='black', line_width=2)
p_growth.y_range.start = 0
p_growth.xgrid.grid_line_color = None
growth_legend = p_growth.legend[0]
growth_legend.location = 'top_center'
growth_legend.orientation = 'horizontal'
growth_legend.click_policy = 'hide'

emit(p_growth, 'logistic_growth_rates')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Optional export

Set `EXPORT_SUMMARY = True` to save the displayed statistics, the plate map, and the selected blank means. `n` is the number of included technical wells at each time point. Changing blank wells, masks or channels requires rerunning the processing cells and plots.

In [15]:
EXPORT_SUMMARY = True
if EXPORT_SUMMARY:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    channel_slug = PROBE_CHANNEL.replace(' ', '_')
    fits_dir = OUTPUT_DIR / 'exp_fits'
    fits_dir.mkdir(parents=True, exist_ok=True)

    # One CSV per biological sample, with separate rows for light, well and reading.
    # Keep every time point; excluded measurements are NaN and marked included=False.
    for sample in SELECTED_SAMPLES:
        sample_frames = []
        for light in LIGHTS:
            idx = sample_indices(sample, light)
            included = masks[light][idx]
            od = processed[light]['od600'][idx]
            n_wells, n_readings = od.shape
            frame = pd.DataFrame({
                'sample': sample,
                'family': sample_family(sample),
                'light': light,
                'well': np.repeat(plate_map.index[idx].to_numpy(), n_readings),
                'channel': PROBE_CHANNEL,
                'blank_control': sample_blank[sample],
                'reading': np.tile(np.arange(n_readings), n_wells),
                # Shared OD/fluorescence time points, as plotted (hours since first OD).
                'hours': np.tile(times[light], n_wells),
                'included': included.ravel(),
                'od600': np.where(included, od, np.nan).ravel(),
                # The corrected, OD-normalised and zero-clipped values used in plots.
                'normalised_fluorescence': np.where(
                    included, processed[light]['normalised_fluorescence'][idx],
                    np.nan).ravel(),
            })
            # Reuse the fits used for fluorescence correction: sample + light,
            # rather than refitting or pooling biological replicates in a family.
            for parameter in ['r', 'Nmax', 'N0']:
                frame[f'logistic_{parameter}'] = logistic_params[sample][light][parameter]
            sample_frames.append(frame)
        target = fits_dir / f'{channel_slug}_{sample}.csv'
        pd.concat(sample_frames, ignore_index=True).to_csv(target, index=False)
        print(f'Saved {target}')

    # Retain the existing summary, plate map and blank exports.
    summary.to_csv(OUTPUT_DIR / f'{channel_slug}_summary.csv', index=False)
    plate_map.to_csv(OUTPUT_DIR / 'plate_map.csv')
    pd.DataFrame([{'light': light, 'blank_control': label, 'channel': channel, 'reading': i,
                   'hours': times[light][i], 'blank_mean': value,
                   'blank_wells': ','.join(blank_wells[light][label])}
                  for light in LIGHTS for label, channels in blanks[light].items()
                  for channel, values in channels.items()
                  for i, value in enumerate(values)]).to_csv(OUTPUT_DIR / 'blank_means.csv', index=False)


PermissionError: [Errno 13] Permission denied: '/mnt/d/CODE/CcaSolitaiRe/Yuhang/26-10-21 Kirill M9/figures_diya_analysis/exp_fits/GFP_395nm_K_D_1.csv'